# Nemotron SFT Smoke Run — v1.1 trace corpus — 2026-04-30

Purpose:
- consume `train_traces_v1_1.jsonl` from Notebook A
- run a **small** SFT smoke test
- prove LoRA training can start and save adapter artifacts
- do **not** optimize leaderboard score here

Success criteria:
- training starts without environment/data formatting failures
- adapter folder is created
- `adapter_config.json` exists
- `adapter_model.safetensors` or equivalent adapter weights exist

This notebook intentionally does **not** create the final `submission.zip`; packaging/validation remains Notebook C.


In [ ]:
import os
import sys
import json
import stat
import shutil
import zipfile
import gc
from pathlib import Path

import pandas as pd

# Keep this tiny. Increase only after the smoke path works.
TRACE_VERSION = "v1_1"
SMOKE_SAMPLE_SIZE = 128
RANDOM_SEED = 42

LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

MAX_SEQ_LEN = 1024
NUM_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 2e-4

WORKING_DIR = Path("/kaggle/working")
TRACE_JSONL_CANDIDATES = [
    WORKING_DIR / "train_traces_v1_1.jsonl",
]

# Also search Kaggle inputs in case v1.1 traces were added as a dataset.
if Path("/kaggle/input").exists():
    TRACE_JSONL_CANDIDATES.extend(Path("/kaggle/input").rglob("train_traces_v1_1.jsonl"))

TRACE_JSONL_PATH = next((p for p in TRACE_JSONL_CANDIDATES if p.exists()), None)
if TRACE_JSONL_PATH is None:
    print("Known candidates:")
    for p in TRACE_JSONL_CANDIDATES[:50]:
        print(" -", p)
    raise FileNotFoundError(
        "Could not find train_traces_v1_1.jsonl. "
        "Run Notebook A first in the same Kaggle session or add the v1.1 trace artifacts as an input dataset."
    )

OUTPUT_DIR = WORKING_DIR / "adapter_smoke_v1_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TRACE_JSONL_PATH:", TRACE_JSONL_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


## Optional offline package setup

This cell uses the offline packages dataset if it is attached. If not attached, it assumes the Kaggle environment already has the needed packages.


In [ ]:
OFFLINE_PACKAGE_DIRS = [
    Path("/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"),
    Path("/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"),
]

offline_dir = next((p for p in OFFLINE_PACKAGE_DIRS if p.exists()), None)

if offline_dir is not None:
    print("Installing offline packages from:", offline_dir)
    # Do not fail silently: if install breaks, we want to see it early.
    !pip install -q --no-index --find-links {offline_dir} datasets trl --ignore-installed
else:
    print("Offline package dir not found; trying imports from current Kaggle environment.")


In [ ]:
import torch
import torch.nn.functional as F

import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

try:
    from trl import SFTTrainer, SFTConfig
    print("TRL import OK.")
except Exception as e:
    raise ImportError(
        "Could not import TRL/SFTTrainer. Attach the offline packages dataset or use a Kaggle image with trl installed."
    ) from e

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Runtime patches used for Nemotron/Mamba/Triton on Kaggle

These patches are conservative and are here only to improve smoke-run robustness. If this still fails, capture the exact traceback before changing hyperparameters.


In [ ]:
# Pure PyTorch rmsnorm fallback. This avoids some Triton ptxas/Blackwell issues.
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast:
        x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None:
        out = out + bias.float()
    if z is not None:
        out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, "rmsnorm_fn"):
        try:
            mod.rmsnorm_fn = _pure_rmsnorm_fn
        except Exception:
            pass

# Copy ptxas-blackwell to a writable temp location if the Kaggle utility script is present.
PTXAS_BLACKWELL_CANDIDATES = [
    Path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"),
    Path("/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script/triton/backends/nvidia/bin/ptxas-blackwell"),
]
ptxas_src = next((p for p in PTXAS_BLACKWELL_CANDIDATES if p.exists()), None)
if ptxas_src is not None:
    dst = Path("/tmp/ptxas-blackwell")
    shutil.copy2(ptxas_src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = str(dst)
    print("Copied ptxas-blackwell to:", dst)
else:
    print("ptxas-blackwell utility binary not found; continuing without copy.")

try:
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
    print("Patched Triton get_ptxas_version.")
except Exception as e:
    print("Could not patch Triton compiler version:", repr(e))


## Load and format v1.1 trace corpus

In [ ]:
records = []
with open(TRACE_JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print("Total trace records:", len(records))
assert records, "No records found in trace JSONL."

df = pd.DataFrame(records)
print(df[["id", "category", "answer", "approx_trace_chars"]].head())
print("\nCategory counts:")
print(df["category"].value_counts())

# Stratified-ish smoke sample: sample across categories so we test all three trace templates.
sampled = (
    df.groupby("category", group_keys=False)
      .apply(lambda x: x.sample(min(len(x), max(1, SMOKE_SAMPLE_SIZE // df["category"].nunique())), random_state=RANDOM_SEED))
      .sample(frac=1.0, random_state=RANDOM_SEED)
      .head(SMOKE_SAMPLE_SIZE)
      .reset_index(drop=True)
)

print("\nSmoke sample shape:", sampled.shape)
print(sampled["category"].value_counts())


In [ ]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
print("MODEL_PATH:", MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def build_training_text(row):
    messages = row["messages"]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=True,
        )
    except TypeError:
        # Some tokenizer versions may not accept enable_thinking.
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        # Fallback: preserve clean user/assistant turn structure.
        user = messages[0]["content"]
        assistant = messages[1]["content"]
        return f"User:\n{user}\n\nAssistant:\n{assistant}{tokenizer.eos_token}"

sampled["text"] = sampled.apply(build_training_text, axis=1)
hf_dataset = Dataset.from_pandas(sampled[["text", "id", "category", "answer"]])

print("Example training text:")
print(hf_dataset[0]["text"][:1200])


## Load model + LoRA

Smoke configuration uses rank 32 to match competition adapter limits, but only a tiny sample. If this model-load/training step fails with OOM or device mismatch, do **not** retry blindly; capture the traceback and adjust loading strategy.


In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Force slow path if Nemotron module exposes the fast-path flag.
for name, mod in list(sys.modules.items()):
    if "modeling_nemotron_h" in name and hasattr(mod, "is_fast_path_available"):
        try:
            mod.is_fast_path_available = False
            print(f"Patched {name}: is_fast_path_available = False")
        except Exception:
            pass

model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## SFT smoke training

The first goal is simply: does one small SFT run complete and save a valid adapter folder?


In [ ]:
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=1,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting SFT smoke training...")
train_result = trainer.train()
print("Training complete.")
print(train_result)


## Save adapter artifacts and validate folder

This does not create the final submission zip. It only proves that a PEFT adapter can be saved.


In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR / "tokenizer_debug")

print("Adapter output files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUTPUT_DIR), f"({p.stat().st_size/1024:.1f} KB)")

adapter_config = OUTPUT_DIR / "adapter_config.json"
adapter_safetensors = OUTPUT_DIR / "adapter_model.safetensors"
adapter_bin = OUTPUT_DIR / "adapter_model.bin"

assert adapter_config.exists(), "Missing adapter_config.json"
assert adapter_safetensors.exists() or adapter_bin.exists(), "Missing adapter model weights"

print("\nSmoke adapter save validation passed.")
print("Use this folder for Notebook C adapter validation, not direct final submission yet:")
print(OUTPUT_DIR)


## Optional smoke zip for transfer/debug only

This zip is for moving the smoke adapter to a validation notebook. It is **not** automatically considered final.


In [ ]:
SMOKE_ZIP_PATH = WORKING_DIR / "adapter_smoke_v1_1.zip"

with zipfile.ZipFile(SMOKE_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.iterdir():
        if p.is_file() and p.name.startswith("adapter_"):
            zf.write(p, arcname=p.name)

print("Created:", SMOKE_ZIP_PATH, f"({SMOKE_ZIP_PATH.stat().st_size/1024/1024:.2f} MB)")
with zipfile.ZipFile(SMOKE_ZIP_PATH, "r") as zf:
    print("Zip contents:", zf.namelist())
    assert "adapter_config.json" in zf.namelist()
